In [33]:
import os
from collections import Counter

import numpy as np
from scipy.sparse import coo_matrix
from sklearn.decomposition import TruncatedSVD
PROJECT_DIR = "/Users/ppers/scientific_work/create-complete-lang/"
BASE = PROJECT_DIR + "data/corpora/marathi/news_leipzig/mar_news_2010_300K"

WORDS_FILE = BASE + "-words.txt"
CO_S_FILE   = BASE + "-co_s.txt"

MAX_VOCAB = 50000
MIN_WORD_ID = 101

N_COMPONENTS = 300



In [34]:
word_ids = []
word_strings = []
word_freqs = []

with open(WORDS_FILE, encoding="utf-8") as f:
    for line in f:
        # формат: word_id word frequency (разделитель — пробел/таб)
        parts = line.rstrip("\n").split()
        if len(parts) != 3:
            continue
        wid_str, word, freq_str = parts
        wid  = int(wid_str)
        freq = int(freq_str)
        word_ids.append(wid)
        word_strings.append(word)
        word_freqs.append(freq)

len(word_ids), word_ids[:5], word_strings[:5], word_freqs[:5]


(290031, [1, 2, 3, 4, 5], ['!', '"', '#', '$', '%'], [5649, 108, 47, 0, 93])

In [35]:
id2index = {}    # word_id → 0..V-1
index2id = []    # индекс → word_id
index2word = []  # индекс → строка слова

for wid, word, freq in zip(word_ids, word_strings, word_freqs):
    if wid < MIN_WORD_ID:
        continue  # пропускаем служебные ID
    if len(index2word) >= MAX_VOCAB:
        break
    idx = len(index2word)
    id2index[wid] = idx
    index2id.append(wid)
    index2word.append(word)

V = len(index2word)
print(f"Размер словаря для SVD: {V}")
print("Примеры слов:", index2word[:20])


Размер словаря для SVD: 50000
Примеры слов: ['आहे', 'या', 'आणि', 'व', 'नाही', 'यांनी', 'आहेत', 'हे', 'तर', 'होते', 'ते', 'असे', 'करण्यात', 'हा', 'ही', 'केली', 'केले', 'काही', 'एक', 'ऑगस्ट']


In [36]:
rows = []
cols = []
data = []

n_lines = 0
n_used  = 0

with open(CO_S_FILE, encoding="utf-8") as f:
    for line in f:
        n_lines += 1
        parts = line.rstrip("\n").split()
        if len(parts) != 4:
            continue
        w1_id_str, w2_id_str, count_str, sig_str = parts

        w1_id = int(w1_id_str)
        w2_id = int(w2_id_str)

        if w1_id not in id2index or w2_id not in id2index:
            continue

        i = id2index[w1_id]
        j = id2index[w2_id]
        count = int(count_str)

        rows.append(i)
        cols.append(j)
        data.append(count)

        rows.append(j)
        cols.append(i)
        data.append(count)

        n_used += 1

print(f"Всего строк в co_s: {n_lines}")
print(f"Использовано пар (до симметрии): {n_used}")
print(f"Ненулевых элементов в матрице (после симметрии): {len(data)}")

X = coo_matrix((data, (rows, cols)), shape=(V, V), dtype=np.float32).tocsr()
X


Всего строк в co_s: 2841216
Использовано пар (до симметрии): 2311292
Ненулевых элементов в матрице (после симметрии): 4622584


<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 2311292 stored elements and shape (50000, 50000)>

In [37]:
svd = TruncatedSVD(
    n_components=N_COMPONENTS,
    algorithm="randomized",
    n_iter=5,
    random_state=0,
)

word_vectors = svd.fit_transform(X)

print("Форма матрицы X:", X.shape)
print("Форма матрицы word_vectors:", word_vectors.shape)


Форма матрицы X: (50000, 50000)
Форма матрицы word_vectors: (50000, 300)


In [38]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# строим отображение слово -> индекс
word2idx = {w: i for i, w in enumerate(index2word)}

def nearest_neighbors(word, topn=10):
    """Возвращает topn ближайших соседей по косинусному сходству для заданного слова."""
    if word not in word2idx:
        print(f"Слова '{word}' нет в словаре (топ-{len(index2word)}).")
        return []
    i = word2idx[word]
    v = word_vectors[i : i+1]  # (1 × d)
    sims = cosine_similarity(v, word_vectors)[0]  # (vocab_size,)

    order = np.argsort(-sims)
    result = []
    for j in order:
        if j == i:
            continue
        result.append((index2word[j], float(sims[j])))
        if len(result) >= topn:
            break
    return result


In [40]:
test_words = [
    "राजा",   # король
    "सरकार", # правительство
    "भारत",  # Индия
]

for w in test_words:
    print("=" * 40)
    print("Слово:", w)
    neighbors = nearest_neighbors(w, topn=3)
    for nb, sim in neighbors:
        print(f"  {nb:15s}  {sim:.3f}")


Слово: राजा
  रसाळ             0.783
  नायक             0.761
  झंझावाती         0.751
Слово: सरकार
  शासन             0.831
  होत              0.827
  न्याय            0.815
Слово: भारत
  दक्षिण           0.888
  समाज             0.881
  अत्यंत           0.881
